# 🌲 Home Credit - Advanced Machine Learning Ensembles (LightGBM & XGBoost)

## 📌 Mục Tiêu So Sánh Mô Hình Học Máy Nâng Cao
Theo chuẩn quy trình `AGENTS.md` (Bước 6 & 7):
1. **So sánh với Mô Hình Học Máy Nâng Cao**: Đào tạo và tinh chỉnh **LightGBM** và **XGBoost** với 5-Fold Cross Validation.
2. **Tối ưu mất cân bằng dữ liệu**: Sử dụng tham số `scale_pos_weight` / `is_unbalance=True` và Early Stopping.
3. **So sánh toàn diện**: So sánh Baseline (Logistic Regression) vs. LightGBM vs. XGBoost trên các thước đo **ROC-AUC**, **PR-AUC**, **KS Statistic**, **Gini**, và **Calibration Curve (Brier Score)**.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, roc_curve, brier_score_loss
from sklearn.calibration import calibration_curve
import lightgbm as lgb
try:
    import xgboost as xgb
except ImportError:
    xgb = None

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

DATA_PATH = Path('../data/processed/home_credit_processed.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/processed/home_credit_processed.csv')

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    RAW_PATH = Path('../data/raw/home-credit-default-risk/application_train.csv')
    if not RAW_PATH.exists():
        RAW_PATH = Path('data/raw/home-credit-default-risk/application_train.csv')
    df = pd.read_csv(RAW_PATH, nrows=50000)
    df['CREDIT_TO_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'] + 1)
    df = pd.get_dummies(df, drop_first=True)
    import re
    df.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', str(col)) for col in df.columns]

import re
df.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', str(col)) for col in df.columns]
X = df.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')
y = df['TARGET']
print(f'✓ Shape dữ liệu đầu vào: X = {X.shape}, Target mean = {y.mean():.4f}')

---
## 1. ⚡ Huấn Luyện LightGBM Mô Hình Cây 5-Fold Stratified K-Fold

In [ ]:
import re
X.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', str(col)) for col in X.columns]

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_lgb = np.zeros(len(df))
feature_importance_df = pd.DataFrame()

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'is_unbalance': True,
    'random_state': 42,
    'verbose': -1
}

print('► Bắt đầu 5-Fold Cross Validation với LightGBM...')
for fold_, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    X_trn, y_trn = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )
    
    oof_lgb[val_idx] = model.predict_proba(X_val)[:, 1]
    fold_auc = roc_auc_score(y_val, oof_lgb[val_idx])
    print(f'  Fold {fold_+1} ROC-AUC: {fold_auc:.4f}')
    
    fold_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': model.feature_importances_
    })
    feature_importance_df = pd.concat([feature_importance_df, fold_importance], axis=0)

cv_auc_lgb = roc_auc_score(y, oof_lgb)
print(f'🎯 Overall LightGBM Out-of-Fold ROC-AUC: {cv_auc_lgb:.4f}')

---
## 2. 📊 Top 20 Đặc Trưng Quan Trọng Nhất Trong LightGBM

In [ ]:
mean_importance = feature_importance_df.groupby('feature')['importance'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(12, 8))
sns.barplot(data=mean_importance.head(20), x='importance', y='feature', palette='Blues_r', hue='feature', legend=False)
plt.title('Top 20 Feature Importance - LightGBM (Alternative Credit Scoring)', fontweight='bold')
plt.xlabel('Importance (Split Count)')
plt.tight_layout()
plt.show()

---
## 3. 🚀 Huấn Luyện XGBoost Mô Hình Ensemble

In [ ]:
oof_xgb = np.zeros(len(df))
scale_pos = (len(y) - sum(y)) / sum(y)

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 600,
    'learning_rate': 0.03,
    'max_depth': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': scale_pos,
    'random_state': 42,
    'n_jobs': -1
}

print('► Bắt đầu 5-Fold Cross Validation với XGBoost...')
for fold_, (trn_idx, val_idx) in enumerate(folds.split(X, y)):
    X_trn, y_trn = X.iloc[trn_idx], y.iloc[trn_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(
        X_trn, y_trn,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    print(f'  Fold {fold_+1} ROC-AUC: {roc_auc_score(y_val, oof_xgb[val_idx]):.4f}')

cv_auc_xgb = roc_auc_score(y, oof_xgb)
print(f'🎯 Overall XGBoost Out-of-Fold ROC-AUC: {cv_auc_xgb:.4f}')

---
## 4. 🏆 Bảng So Sánh Hiệu Năng Mô Hình & Reliability Calibration Plot

In [ ]:
def get_metrics_summary(name, y_true, y_prob):
    auc_val = roc_auc_score(y_true, y_prob)
    gini_val = 2 * auc_val - 1
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks_val = np.max(tpr - fpr) * 100
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc_val = auc(recall, precision)
    brier = brier_score_loss(y_true, y_prob)
    return {
        'Model': name,
        'ROC-AUC': round(auc_val, 4),
        'Gini': round(gini_val, 4),
        'KS Statistic (%)': round(ks_val, 2),
        'PR-AUC': round(pr_auc_val, 4),
        'Brier Score': round(brier, 4)
    }

res_lgb = get_metrics_summary('LightGBM (Ensemble)', y, oof_lgb)
res_xgb = get_metrics_summary('XGBoost (Ensemble)', y, oof_xgb)

comparison_df = pd.DataFrame([res_lgb, res_xgb])
print('=== BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH HỌC MÁY NÂNG CAO ===')
print(comparison_df.to_string(index=False))

# Trực quan hóa Calibration Curve
plt.figure(figsize=(8, 6))
fraction_of_pos_lgb, mean_pred_value_lgb = calibration_curve(y, oof_lgb, n_bins=10)
fraction_of_pos_xgb, mean_pred_value_xgb = calibration_curve(y, oof_xgb, n_bins=10)

plt.plot(mean_pred_value_lgb, fraction_of_pos_lgb, 's-', label='LightGBM', color='#2a9d8f')
plt.plot(mean_pred_value_xgb, fraction_of_pos_xgb, 'o-', label='XGBoost', color='#e76f51')
plt.plot([0, 1], [0, 1], 'k--', label='Perfectly Calibrated')
plt.title('Biểu Đồ Hiệu Chỉnh Xác Suất (Probability Calibration Curve)', fontweight='bold')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.legend()
plt.tight_layout()
plt.show()